# Абстракция в ООП — как я это поняла

## Главная идея

Абстрактный метод объявляется в базовом классе **пустым** (`pass`), чтобы задать **обязательство**: каждый подкласс **обязан** реализовать этот метод по-своему.

Сам базовый класс метод не выполняет — он только говорит: *«у меня есть такой метод, и у всех моих потомков он тоже должен быть»*.

## Аналогия

Представь контракт на работу: в нём написано *«сотрудник обязан уметь готовить отчёты»* — но сам контракт отчёты не готовит.

Реальный отчёт готовит **конкретный человек**, который подписал этот контракт.

- `Product` — это **контракт**
- `PhysicalProduct` / `DigitalProduct` — это **конкретные исполнители**

## Шаг 1 — В базовом классе: только объявление

In [ ]:
from abc import ABC, abstractmethod

class Product(ABC):
    """Абстрактный базовый класс — задаёт контракт."""

    def __init__(self, name: str, price: float):
        self._name = name
        self._price = price

    # Объявлен, но НЕ реализован — только pass
    # Это обязательство для всех подклассов
    @abstractmethod
    def calculate_shipping(self) -> float:
        pass

    @abstractmethod
    def get_info(self) -> str:
        pass

## Шаг 2 — Product нельзя создать напрямую

Именно поэтому в переменной `product` **никогда** не лежит сам `Product` — туда всегда попадает конкретный подкласс.

In [ ]:
try:
    p = Product("Test", 5.0)
except TypeError as e:
    print(f"Ошибка: {e}")

# Вывод: Can't instantiate abstract class Product
# without an implementation for abstract methods ...

## Шаг 3 — В подклассах: реальная реализация

In [ ]:
class PhysicalProduct(Product):
    """Физический товар — доставка считается по весу."""

    def __init__(self, name: str, price: float, weight_kg: float):
        super().__init__(name, price)
        self._weight_kg = weight_kg

    # Реальная реализация — вот что происходит
    def calculate_shipping(self) -> float:
        return round(self._weight_kg * 2.50, 2)

    def get_info(self) -> str:
        return f"[Physical] {self._name} | ${self._price} | {self._weight_kg} kg"


class DigitalProduct(Product):
    """Цифровой товар — доставки нет."""

    def __init__(self, name: str, price: float, download_url: str):
        super().__init__(name, price)
        self._download_url = download_url

    # Другая реальная реализация того же метода
    def calculate_shipping(self) -> float:
        return 0.0

    def get_info(self) -> str:
        return f"[Digital] {self._name} | ${self._price} | {self._download_url}"

## Шаг 4 — Как Python решает, какую версию вызвать

Когда где-то в коде написано `product.calculate_shipping()` — Python смотрит, **что реально лежит** в этой переменной, и вызывает нужную версию автоматически.

In [ ]:
laptop = PhysicalProduct("Laptop", 999.99, weight_kg=2.5)
ebook  = DigitalProduct("Python Guide", 19.99, download_url="http://example.com")

catalog = [laptop, ebook]  # оба хранятся как Product, но реально разные типы

for product in catalog:
    # Один и тот же вызов — разный результат в зависимости от типа объекта
    print(product.get_info())
    print(f"  Shipping: ${product.calculate_shipping()}")
    print()

## Шаг 5 — Если забыть реализовать метод в подклассе

In [ ]:
class BrokenProduct(Product):
    # Забыли написать calculate_shipping() и get_info()
    pass

try:
    b = BrokenProduct("Test", 5.0)
except TypeError as e:
    print(f"Ошибка: {e}")

# Python сразу говорит об ошибке — абстракция не даёт создать
# объект с незаполненным контрактом

## Итог

| Где | Что происходит |
|---|---|
| `Product` (базовый класс) | Метод объявлен как `@abstractmethod` с `pass` — это **контракт** |
| `PhysicalProduct` | Метод **реализован** с формулой по весу |
| `DigitalProduct` | Метод **реализован** как `return 0.0` |
| `cart.py`, `store.py` | Метод **вызывается** — Python сам выбирает нужную версию |
| `Product()` напрямую | **Невозможно** — Python выдаст `TypeError` |

**Главный вывод:** `Product.calculate_shipping()` с `pass` **никогда не вызывается**.  
Он существует только как обязательство: *«все мои подклассы должны реализовать этот метод»*.

## Дополнение: что такое фабрика (Factory)

### Простая идея

Фабрика в ООП — это класс, который **создает объекты**.

Почему так называется:
- Как обычная фабрика производит разные товары по одному процессу,
- так и программная фабрика производит разные объекты по одному входу.

Вместо того чтобы писать по всему коду:
- `PhysicalProduct(...)`
- `DigitalProduct(...)`

мы пишем один вызов:
- `ProductFactory.create("physical", ...)`
- `ProductFactory.create("digital", ...)`

### Зачем нужна фабрика

1. Централизует создание объектов в одном месте.
2. Скрывает детали создания от остального кода.
3. Делает код чище и легче для расширения.
4. Снижает шанс ошибок при ручном создании объектов.

### Почему не "основной движок"

Название `Factory` — стандартный термин в ООП.
Если разработчик видит `ProductFactory`, он сразу понимает:
этот класс **создает продукты**, а не управляет всей бизнес-логикой приложения.

`Store` управляет магазином, `Cart` — корзиной, а `ProductFactory` — только созданием продуктов.

In [ ]:
# Мини-пример фабрики из логики проекта

class ProductFactoryDemo:
    @staticmethod
    def create(product_type: str, **kwargs):
        pt = product_type.lower()
        if pt == "physical":
            return {"type": "PhysicalProduct", **kwargs}
        if pt == "digital":
            return {"type": "DigitalProduct", **kwargs}
        raise ValueError("Unknown product type")


p1 = ProductFactoryDemo.create("physical", name="Laptop", price=999.99, weight_kg=2.5)
p2 = ProductFactoryDemo.create("digital", name="Python Guide", price=19.99, download_url="http://example.com")

print(p1)
print(p2)